# 13 — Project Future Demand and Flexibility

## Research purpose

Apply the six IEA demand endpoints for 2030 and 2035 to the current-plus-future workload results for both candidate-hour settings. Scale the completed results uniformly while retaining allocation methods, workload cases, shift modes and historical weather years.


## Imports

Load the libraries used to validate allocation inputs and calculate demand projections.


In [1]:
from pathlib import Path
from calendar import isleap
import hashlib
import json
from itertools import product
from io import StringIO

import numpy as np
import pandas as pd

## Project location and paths

Locate the project folder and define the paths for the demand metadata, facility and campus tables, allocation summaries and completed partitions. Define separate six-hour, nine-hour and combined projection exports.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_MANUAL = PROJECT_ROOT / "data" / "manual"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_FINAL = PROJECT_ROOT / "data" / "final"
SIX_HOUR_FINAL_DIR = DATA_FINAL / "candidate_hours_6h"
SCENARIO_METADATA_PATH = DATA_MANUAL / "demand_projection_scenario_metadata.csv"
CURRENT_FACILITIES_PATH = DATA_FINAL / "hyperscale_facilities_final.csv"
CURRENT_PLUS_FUTURE_CAMPUSES_PATH = DATA_FINAL / "hyperscale_campuses_current_plus_future_scenario.csv"
PROJECTION_SCENARIOS_PATH = DATA_FINAL / "future_demand_projection_scenarios.csv"
BASE_OPERATOR_YEAR_PATH = DATA_FINAL / "candidate_hours_comparison_operator_year_summary.csv"
QUERY_LOCATIONS_PATH = DATA_INTERIM / "renewables_ninja_query_locations.csv"
BASE_PARQUET_DIRS = {
    (6, "daily_spatiotemporal"): DATA_INTERIM / "operator_daily_spatiotemporal_shift_allocations_6h_by_scenario_parquet",
    (6, "same_hour_geographic"): DATA_INTERIM / "operator_same_hour_geographic_shift_allocations_6h_by_scenario_parquet",
    (9, "daily_spatiotemporal"): DATA_INTERIM / "operator_daily_spatiotemporal_shift_allocations_9h_by_scenario_parquet",
    (9, "same_hour_geographic"): DATA_INTERIM / "operator_same_hour_geographic_shift_allocations_9h_by_scenario_parquet",
}
NINE_HOUR_FINAL_DIR = DATA_FINAL / "candidate_hours_9h"
COMBINED_FINAL_DIR = DATA_FINAL
OPERATOR_OUTPUT_PATH = SIX_HOUR_FINAL_DIR / "projected_flexibility_operator_weather_year_summary.csv"
WEATHER_OUTPUT_PATH = SIX_HOUR_FINAL_DIR / "projected_flexibility_weather_year_summary.csv"
SUMMARY_OUTPUT_PATH = SIX_HOUR_FINAL_DIR / "projected_flexibility_summary.csv"
REGION_WEATHER_OUTPUT_PATH = SIX_HOUR_FINAL_DIR / "projected_flexibility_region_pair_weather_year_summary.csv"
REGION_OUTPUT_PATH = SIX_HOUR_FINAL_DIR / "projected_flexibility_region_pair_summary.csv"

## Projection settings

Define both candidate-hour settings and allocation methods, the nine workload cases, two shift modes and three weather years. Retain Amazon, Google and Microsoft and identify LP as the primary allocation method.


In [3]:
CANDIDATE_HOURS = [6, 9]
ALLOCATION_METHODS = ["greedy", "optimised_lp"]
SHIFT_MODES = ["daily_spatiotemporal", "same_hour_geographic"]
UTILISATION_IDS = ["util_80", "util_85", "util_90"]
FLEXIBILITY_IDS = ["flex_20", "flex_25", "flex_30"]
WEATHER_YEARS = [2023, 2024, 2025]
OPERATORS = ["Amazon", "Google", "Microsoft"]
PRIMARY_ALLOCATION_METHOD = "optimised_lp"
SCENARIO_METADATA_COLUMNS = [
    "utilisation_scenario_id", "base_utilisation", "utilisation_scenario_label",
    "is_central_utilisation", "flexibility_scenario_id", "flexible_workload_share",
    "scenario_label", "is_central_flexibility", "is_central",
]

## Check complete case identities

Define a check that requires each expected case exactly once. Compare the full combination of case identifiers to detect missing, duplicate or unexpected records.


In [4]:
def require_grid(frame, domains, label):
    """Require every declared key once; row counts alone cannot detect wrong cases."""
    keys = list(domains)
    if frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError(f"{label}: missing or duplicate identity keys")
    expected = pd.MultiIndex.from_product(list(domains.values()), names=keys)
    actual = pd.MultiIndex.from_frame(frame[keys])
    missing, extra = expected.difference(actual), actual.difference(expected)
    if len(missing) or len(extra):
        raise ValueError(f"{label}: {len(missing)} missing and {len(extra)} unexpected cases")

## Read IEA capacity inputs

Read the six demand endpoints and common 2024 capacity baseline from the manual CSV, transcribed from the World Data annex of [IEA (2025), Energy and AI](https://www.iea.org/reports/energy-and-ai). Check that each demand scenario appears in both target years, capacities are positive and complete, and only the moderate scenario is marked central.


In [5]:
metadata = pd.read_csv(
    SCENARIO_METADATA_PATH,
    dtype={
        "target_iea_hyperscale_capacity_gw": "float64",
        "iea_baseline_hyperscale_2024_gw": "float64",
        "is_central_demand_scenario": "boolean",
    },
)

required_metadata = {
    "demand_scenario_id", "demand_scenario_label", "iea_scenario_name", "target_year",
    "target_iea_hyperscale_capacity_gw", "iea_baseline_hyperscale_2024_gw",
    "is_central_demand_scenario",
}

missing = sorted(required_metadata - set(metadata.columns))
if missing:
    raise ValueError(f"IEA capacity input CSV is missing columns: {missing}")
if metadata[sorted(required_metadata)].isna().any().any():
    raise ValueError("IEA capacity inputs and scenario labels must be complete")
require_grid(metadata, {
    "demand_scenario_id": ["conservative", "moderate", "accelerated"],
    "target_year": [2030, 2035],
}, "IEA capacity inputs")

capacity_columns = ["target_iea_hyperscale_capacity_gw", "iea_baseline_hyperscale_2024_gw"]
if not np.isfinite(metadata[capacity_columns]).all().all() or metadata[capacity_columns].le(0).any().any():
    raise ValueError("IEA target and baseline capacities must be finite and positive")
if metadata["iea_baseline_hyperscale_2024_gw"].nunique() != 1:
    raise ValueError("All IEA endpoints must use the same 2024 baseline")
if not metadata["is_central_demand_scenario"].eq(metadata["demand_scenario_id"].eq("moderate")).all():
    raise ValueError("Only the moderate IEA demand scenario can be central")

metadata

,demand_scenario_id,demand_scenario_label,iea_scenario_name,target_year,target_iea_hyperscale_capacity_gw,iea_baseline_hyperscale_2024_gw,is_central_demand_scenario
0,conservative,Conservative demand growth,Headwinds,2030,62.0,36.0,False
1,conservative,Conservative demand growth,Headwinds,2035,64.0,36.0,False
2,moderate,Moderate demand growth,Base,2030,85.0,36.0,True
3,moderate,Moderate demand growth,Base,2035,103.0,36.0,True
4,accelerated,Accelerated demand growth,Lift-Off,2030,108.0,36.0,False
5,accelerated,Accelerated demand growth,Lift-Off,2035,139.0,36.0,False


## Calculate endpoint scale factors

Multiply current operational capacity by each IEA target-to-baseline ratio and use represented current-plus-future capacity as the lower bound. Divide the resulting projected capacity by current-plus-future capacity to obtain a common scaling factor. This scales the represented network without adding new campus locations.


In [6]:
current = pd.read_csv(CURRENT_FACILITIES_PATH)
campuses = pd.read_csv(CURRENT_PLUS_FUTURE_CAMPUSES_PATH)
current_operational_capacity_mw = current["capacity_mw"].sum()
current_plus_future_capacity_mw = campuses["capacity_mw"].sum()

scenarios = metadata.copy()
scenarios["projection_scenario_id"] = scenarios["demand_scenario_id"] + "_" + scenarios["target_year"].astype(str)
scenarios["iea_growth_multiplier"] = scenarios["target_iea_hyperscale_capacity_gw"] / scenarios["iea_baseline_hyperscale_2024_gw"]
scenarios["scoped_current_operational_capacity_mw"] = current_operational_capacity_mw
scenarios["current_plus_future_lower_bound_capacity_mw"] = current_plus_future_capacity_mw
scenarios["projected_scoped_capacity_mw"] = current_operational_capacity_mw * scenarios["iea_growth_multiplier"]
scenarios["additional_growth_mw"] = (scenarios["projected_scoped_capacity_mw"] - current_plus_future_capacity_mw).clip(lower=0)
scenarios["projected_total_capacity_mw"] = current_plus_future_capacity_mw + scenarios["additional_growth_mw"]
scenarios["capacity_scale_factor_vs_current_plus_future"] = scenarios["projected_total_capacity_mw"] / current_plus_future_capacity_mw
scenarios["projection_method"] = "iea_global_hyperscale_installed_capacity_scaled_to_scoped_current_operational_baseline"
scenarios["growth_allocation_method"] = "uniform_scalar_scaling_of_current_plus_future_outputs_no_spatial_reallocation"
scenarios["source_file"] = str(SCENARIO_METADATA_PATH.relative_to(PROJECT_ROOT))

scenarios = scenarios.sort_values(["target_year", "target_iea_hyperscale_capacity_gw"]).reset_index(drop=True)
if len(scenarios) != 6:
    raise ValueError("Expected six demand endpoint scenarios")
scenarios = scenarios[[
    "demand_scenario_id", "demand_scenario_label", "iea_scenario_name", "target_year",
    "target_iea_hyperscale_capacity_gw", "is_central_demand_scenario",
    "projection_scenario_id", "iea_baseline_hyperscale_2024_gw", "iea_growth_multiplier",
    "scoped_current_operational_capacity_mw", "current_plus_future_lower_bound_capacity_mw",
    "projected_scoped_capacity_mw", "additional_growth_mw", "projected_total_capacity_mw",
    "capacity_scale_factor_vs_current_plus_future", "projection_method", "growth_allocation_method",
    "source_file",
]]

# Export only after the complete projection validation below.
print(scenarios[["projection_scenario_id", "target_iea_hyperscale_capacity_gw", "capacity_scale_factor_vs_current_plus_future"]].to_string(index=False))

projection_scenario_id  target_iea_hyperscale_capacity_gw  capacity_scale_factor_vs_current_plus_future
     conservative_2030                               62.0                                      1.380891
         moderate_2030                               85.0                                      1.893157
      accelerated_2030                              108.0                                      2.405423
     conservative_2035                               64.0                                      1.425436
         moderate_2035                              103.0                                      2.294061
      accelerated_2035                              139.0                                      3.095869


## Prepare projection metadata

Select the scenario identifiers, target capacities and scaling factors carried through each projected result. Use the CSV-parsed numeric representation consistently for the projection calculations.


In [7]:
# Use the CSV-parsed numeric representation consistently for projection arithmetic.
projections = pd.read_csv(StringIO(scenarios.to_csv(index=False)))
required_projection = {
    "projection_scenario_id", "demand_scenario_id", "demand_scenario_label", "iea_scenario_name",
    "target_year", "target_iea_hyperscale_capacity_gw", "projected_total_capacity_mw",
    "capacity_scale_factor_vs_current_plus_future",
}

missing = sorted(required_projection - set(projections.columns))
if missing:
    raise ValueError(f"Projection table is missing columns: {missing}")
if len(projections) != 6:
    raise ValueError("Expected three demand scenarios for each of 2030 and 2035")
projection_metadata = [
    "projection_scenario_id", "demand_scenario_id", "demand_scenario_label", "iea_scenario_name",
    "target_year", "target_iea_hyperscale_capacity_gw", "projected_total_capacity_mw",
    "capacity_scale_factor_vs_current_plus_future",
]

projection_cross = projections[projection_metadata].copy()

## Validate allocation summaries

Read the combined operator summary and select the current-plus-future network. Check the required quantities, complete case identities and reference-case flags, then confirm that alternative hour settings, methods, flexibility shares and shift modes use the same demand and capacity baseline.


In [8]:
base_operator = pd.read_csv(BASE_OPERATOR_YEAR_PATH)
required_base = {
    "candidate_hours", "allocation_method", "model_layer", "operator_group", "weather_year", "shift_mode",
    "shifted_mwh", "total_actual_load_mwh", "total_capacity_hours_mwh",
    "total_weighted_renewable_gain", "total_weighted_abs_time_hours", "allocation_count",
    "capped_hour_count", "hourly_observation_count",
    *SCENARIO_METADATA_COLUMNS,
}

missing = sorted(required_base - set(base_operator.columns))
if missing:
    raise ValueError(f"Base operator summary is missing columns: {missing}")

base_operator = base_operator[base_operator["model_layer"].eq("current_plus_future")].copy()
if sorted(base_operator["allocation_method"].unique()) != ALLOCATION_METHODS:
    raise ValueError("Expected exactly greedy and optimised_lp base results")

case_domains = {
    "candidate_hours": CANDIDATE_HOURS, "allocation_method": ALLOCATION_METHODS,
    "utilisation_scenario_id": UTILISATION_IDS, "flexibility_scenario_id": FLEXIBILITY_IDS,
    "shift_mode": SHIFT_MODES, "model_layer": ["current_plus_future"],
}
require_grid(base_operator, {**case_domains, "operator_group": OPERATORS, "weather_year": WEATHER_YEARS}, "Base operator summary")

central = base_operator["utilisation_scenario_id"].eq("util_85") & base_operator["flexibility_scenario_id"].eq("flex_25")
if not base_operator["is_central"].eq(central).all():
    raise ValueError("Only the 85%/25% workload case can be central")

# Demand/capacity are shared physical baselines, not additional energy for each alternative.
for column in ["total_actual_load_mwh", "total_capacity_hours_mwh"]:
    spread = base_operator.groupby(["utilisation_scenario_id", "operator_group", "weather_year"])[column].agg(lambda values: values.max() - values.min())
    if spread.isna().any() or spread.max() > 1e-6:
        raise ValueError(f"{column} differs across candidate hours, allocators, flexibility or shift modes")

## Reconcile campus and model capacity

Divide each weather year's capacity-hour total by its calendar hours, including leap years, to recover the capacity represented in the allocation model. Check that this matches the campus capacity used to calculate the projection factors.


In [9]:
if not np.isfinite(current_plus_future_capacity_mw) or current_plus_future_capacity_mw <= 0:
    raise ValueError("Current-plus-future campus capacity must be finite and positive")
capacity_check_keys = [
    "candidate_hours", "allocation_method", "utilisation_scenario_id",
    "flexibility_scenario_id", "shift_mode", "weather_year",
]

capacity_check = base_operator.groupby(capacity_check_keys, as_index=False)["total_capacity_hours_mwh"].sum()
year_hours = capacity_check["weather_year"].map(lambda year: 24 * (366 if isleap(int(year)) else 365))
capacity_check["implied_capacity_mw"] = capacity_check["total_capacity_hours_mwh"] / year_hours
if not np.isclose(
    capacity_check["implied_capacity_mw"], current_plus_future_capacity_mw, rtol=1e-12, atol=1e-6,
).all():
    raise ValueError("Allocation-summary capacity does not match the current-plus-future campus capacity")

## Define file checksum verification

Define a SHA-256 checksum function for the completed allocation files. Read each file incrementally so checksum verification does not require loading its entire contents into memory.


In [10]:
def file_sha256(path):
    """Hash an input file without loading its bytes into memory."""
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

## Identify completed allocation inputs

Define a reader that lists every expected method and workload partition for a candidate-hour setting and shift mode. Check root and completion identities and verify each allocation file's checksum before accepting it.


In [11]:
def completed_partitions(
    directory, candidate_hours, shift_mode, *, allocation_methods, utilisation_ids, flexibility_ids
):
    """Enumerate completed declared combinations, excluding temporary or orphan files."""
    root_path = directory / "_allocation_root_manifest.json"
    if not root_path.exists():
        raise FileNotFoundError(f"Missing completed notebook-12 allocation root: {root_path}")
    root = json.loads(root_path.read_text())
    if root.get("shift_mode") != shift_mode or root.get("candidate_hours", 6) != candidate_hours:
        raise ValueError(f"Candidate-hour/shift-mode root mismatch: {directory}")
    if candidate_hours == 9 and root.get("eligibility_rule") != "frozen-daily-ranks-v1":
        raise ValueError(f"Unexpected nine-hour ranking contract: {directory}")
    records = []
    for method, utilisation, flexibility in product(allocation_methods, utilisation_ids, flexibility_ids):
        partition = directory / f"method_{method}" / f"util_{utilisation}" / f"flex_{flexibility}"
        success_path, detail_path = partition / "_SUCCESS.json", partition / "part-00000.parquet"
        if not success_path.exists() or not detail_path.exists():
            raise FileNotFoundError(f"Incomplete notebook-12 allocation partition: {partition}")
        success = json.loads(success_path.read_text())
        expected = {"allocation_method": method, "utilisation_scenario_id": utilisation,
                    "flexibility_scenario_id": flexibility, "shift_mode": shift_mode}
        if any(success.get(key) != value for key, value in expected.items()):
            raise ValueError(f"Allocation partition identity mismatch: {partition}")
        if success.get("candidate_hours", 6) != candidate_hours:
            raise ValueError(f"Candidate hours do not match the allocation partition: {partition}")
        if success.get("implementation_id") != root.get("implementation_id"):
            raise ValueError(f"Allocation implementation does not match its root: {partition}")
        if candidate_hours == 9 and success.get("eligibility_rule") != root.get("eligibility_rule"):
            raise ValueError(f"Allocation ranking rule does not match its root: {partition}")
        if file_sha256(detail_path) != success.get("detail_sha256"):
            raise ValueError(f"Allocation checksum mismatch: {detail_path}")
        records.append((detail_path, success))
    return root, records

## Verify all allocation roots

Check the completed partitions for all four allocation roots. Confirm that both candidate-hour settings and shift modes share the same hourly inputs, workload definitions and model implementation.


In [12]:
# All completed allocation roots must identify the same model inputs.
allocation_inputs = {}
reference_contract = None
for (candidate_hours, shift_mode), directory in BASE_PARQUET_DIRS.items():
    root, partitions = completed_partitions(
        directory, candidate_hours, shift_mode,
        allocation_methods=ALLOCATION_METHODS,
        utilisation_ids=UTILISATION_IDS,
        flexibility_ids=FLEXIBILITY_IDS,
    )
    signature = {key: root.get(key) for key in [
        "hourly_input_sha256", "utilisation_scenarios_sha256", "flexibility_scenarios_sha256", "implementation_id",
    ]}
    if not all(signature.values()) or (reference_contract is not None and signature != reference_contract):
        raise ValueError(f"Six/nine-hour allocations do not share the same model inputs: {directory}")
    reference_contract = signature
    allocation_inputs[(candidate_hours, shift_mode)] = partitions
    print(f"Verified {candidate_hours}h {shift_mode}: {len(partitions)} completed allocation partitions")

Verified 6h daily_spatiotemporal: 18 completed allocation partitions
Verified 6h same_hour_geographic: 18 completed allocation partitions
Verified 9h daily_spatiotemporal: 18 completed allocation partitions
Verified 9h same_hour_geographic: 18 completed allocation partitions


## Scale the demand baseline

Select one daily-shifting and 20% flexibility slice to represent demand for each hour setting, method, utilisation and weather year. Confirm that these alternatives share the same demand baseline and multiply it by each capacity factor.


In [13]:
# Demand is hour-setting/allocator-independent. Keep one labelled slice per alternative; never sum alternatives.
base_demand = (
    base_operator[
        base_operator["shift_mode"].eq("daily_spatiotemporal")
        & base_operator["flexibility_scenario_id"].eq("flex_20")
    ]
    .groupby(["candidate_hours", "allocation_method", "utilisation_scenario_id", "base_utilisation", "weather_year"], as_index=False)["total_actual_load_mwh"].sum()
    .rename(columns={"total_actual_load_mwh": "baseline_current_plus_future_total_demand_mwh"})
)

demand_check = base_demand.pivot_table(
    index=["utilisation_scenario_id", "weather_year"],
    columns=["candidate_hours", "allocation_method"], values="baseline_current_plus_future_total_demand_mwh", aggfunc="first",
)
if demand_check.isna().any().any() or (demand_check.max(axis=1) - demand_check.min(axis=1)).max() > 1e-6:
    raise ValueError("Demand baselines differ by candidate hours or allocation method")

projected_demand = base_demand.merge(projection_cross, how="cross")
projected_demand["projected_total_demand_mwh"] = (
    projected_demand["baseline_current_plus_future_total_demand_mwh"]
    * projected_demand["capacity_scale_factor_vs_current_plus_future"]
)

projected_demand["projected_total_demand_twh"] = projected_demand["projected_total_demand_mwh"] / 1_000_000

## Scale operator flows

Multiply each operator's shifted energy and weighted renewable and time totals by the capacity factor, then recalculate the weighted metrics where shifted energy is positive. Calculate each operator's shifted share against total projected demand and mark the primary allocation method. Retain allocation counts, base load and daily-summary fields as unscaled diagnostics.


In [14]:
operator_projected = base_operator.merge(projection_cross, how="cross")
operator_projected["baseline_shifted_mwh"] = operator_projected["shifted_mwh"]
operator_projected["baseline_total_weighted_renewable_gain"] = operator_projected["total_weighted_renewable_gain"]
operator_projected["baseline_total_weighted_abs_time_hours"] = operator_projected["total_weighted_abs_time_hours"]
scale = operator_projected["capacity_scale_factor_vs_current_plus_future"]
operator_projected["shifted_mwh"] = operator_projected["baseline_shifted_mwh"] * scale
operator_projected["total_weighted_renewable_gain"] = operator_projected["baseline_total_weighted_renewable_gain"] * scale
operator_projected["total_weighted_abs_time_hours"] = operator_projected["baseline_total_weighted_abs_time_hours"] * scale
operator_projected["mwh_weighted_renewable_availability_gain"] = np.divide(
    operator_projected["total_weighted_renewable_gain"], operator_projected["shifted_mwh"],
    out=np.full(len(operator_projected), np.nan), where=operator_projected["shifted_mwh"].to_numpy() > 0,
)
operator_projected["mwh_weighted_absolute_time_difference_hours"] = np.divide(
    operator_projected["total_weighted_abs_time_hours"], operator_projected["shifted_mwh"],
    out=np.full(len(operator_projected), np.nan), where=operator_projected["shifted_mwh"].to_numpy() > 0,
)
operator_projected = operator_projected.merge(
    projected_demand[[
        "candidate_hours", "allocation_method", "projection_scenario_id", "utilisation_scenario_id", "weather_year",
        "projected_total_demand_mwh", "projected_total_demand_twh",
    ]],
    on=["candidate_hours", "allocation_method", "projection_scenario_id", "utilisation_scenario_id", "weather_year"], how="left", validate="many_to_one",
)
operator_projected["shifted_share_of_projected_demand"] = operator_projected["shifted_mwh"] / operator_projected["projected_total_demand_mwh"]
operator_projected["is_primary_allocation_method"] = operator_projected["allocation_method"].eq(PRIMARY_ALLOCATION_METHOD)
operator_projected["projection_method"] = "uniform_capacity_scaling_from_method_specific_current_plus_future_allocations"

## Aggregate projected weather-year results

Sum projected flows and weighted totals across operators for each matched case and weather year, retaining the common projected demand once. Calculate shifted shares and weighted metrics, and derive realised base utilisation from the unscaled load and capacity-hour totals.


In [15]:
weather_group_cols = projection_metadata + SCENARIO_METADATA_COLUMNS + ["candidate_hours", "allocation_method", "model_layer", "shift_mode", "weather_year"]
weather = (
    operator_projected.groupby(weather_group_cols, dropna=False)
    .agg(
        operators=("operator_group", "nunique"),
        allocation_count=("allocation_count", "sum"),
        shifted_mwh=("shifted_mwh", "sum"),
        total_weighted_renewable_gain=("total_weighted_renewable_gain", "sum"),
        total_weighted_abs_time_hours=("total_weighted_abs_time_hours", "sum"),
        projected_total_demand_mwh=("projected_total_demand_mwh", "first"),
        projected_total_demand_twh=("projected_total_demand_twh", "first"),
        total_actual_load_mwh=("total_actual_load_mwh", "sum"),
        total_capacity_hours_mwh=("total_capacity_hours_mwh", "sum"),
        capped_hour_count=("capped_hour_count", "sum"),
        hourly_observation_count=("hourly_observation_count", "sum"),
    )
    .reset_index()
)
weather["shifted_gwh"] = weather["shifted_mwh"] / 1_000
weather["shifted_twh"] = weather["shifted_mwh"] / 1_000_000
weather["shifted_share_of_projected_demand"] = weather["shifted_mwh"] / weather["projected_total_demand_mwh"]
weather["mwh_weighted_renewable_availability_gain"] = np.divide(
    weather["total_weighted_renewable_gain"], weather["shifted_mwh"],
    out=np.full(len(weather), np.nan), where=weather["shifted_mwh"].to_numpy() > 0,
)
weather["mwh_weighted_absolute_time_difference_hours"] = np.divide(
    weather["total_weighted_abs_time_hours"], weather["shifted_mwh"],
    out=np.full(len(weather), np.nan), where=weather["shifted_mwh"].to_numpy() > 0,
)
weather["realised_base_utilisation"] = weather["total_actual_load_mwh"] / weather["total_capacity_hours_mwh"]
weather["is_primary_allocation_method"] = weather["allocation_method"].eq(PRIMARY_ALLOCATION_METHOD)

## Compare same-hour and daily shifting

Divide same-hour shifted energy by daily shifted energy within each matching hour setting, method, projection, workload case and weather year. Attach this ratio to both shift-mode results for comparison.


In [16]:
ratio_keys = ["candidate_hours", "allocation_method", "projection_scenario_id", "utilisation_scenario_id", "flexibility_scenario_id", "weather_year"]
ratio = weather.pivot_table(index=ratio_keys, columns="shift_mode", values="shifted_mwh", aggfunc="sum").reset_index()
ratio["same_hour_share_of_daily_spatiotemporal"] = ratio["same_hour_geographic"] / ratio["daily_spatiotemporal"]
weather = weather.merge(ratio[ratio_keys + ["same_hour_share_of_daily_spatiotemporal"]], on=ratio_keys, how="left")

## Summarise historical weather sensitivity

Calculate medians, minima and maxima across the three historical weather years for each projection case. Use the range to describe sensitivity to weather conditions.


In [17]:
summary_group_cols = projection_metadata + SCENARIO_METADATA_COLUMNS + ["candidate_hours", "allocation_method", "model_layer", "shift_mode"]
summary = (
    weather.groupby(summary_group_cols, dropna=False)
    .agg(
        weather_year_count=("weather_year", "nunique"),
        shifted_gwh_median=("shifted_gwh", "median"),
        shifted_gwh_min_weather_year=("shifted_gwh", "min"),
        shifted_gwh_max_weather_year=("shifted_gwh", "max"),
        shifted_twh_median=("shifted_twh", "median"),
        shifted_share_of_projected_demand_median=("shifted_share_of_projected_demand", "median"),
        shifted_share_of_projected_demand_min_weather_year=("shifted_share_of_projected_demand", "min"),
        shifted_share_of_projected_demand_max_weather_year=("shifted_share_of_projected_demand", "max"),
        projected_total_demand_twh_median=("projected_total_demand_twh", "median"),
        same_hour_share_of_daily_spatiotemporal_median=("same_hour_share_of_daily_spatiotemporal", "median"),
        mwh_weighted_renewable_availability_gain_median=("mwh_weighted_renewable_availability_gain", "median"),
        mwh_weighted_absolute_time_difference_hours_median=("mwh_weighted_absolute_time_difference_hours", "median"),
        realised_base_utilisation_median=("realised_base_utilisation", "median"),
        weather_years_used=("weather_year", lambda values: "|".join(map(str, sorted(set(values))))),
    )
    .reset_index()
)
summary["is_primary_allocation_method"] = summary["allocation_method"].eq(PRIMARY_ALLOCATION_METHOD)

print(f"Projection summary cases: {len(summary):,}")

Projection summary cases: 432


## Map query locations to world regions

Read the world-region label for each renewable query location. Check that the study contains the five regions used to summarise directed flows.


In [18]:
locations = pd.read_csv(QUERY_LOCATIONS_PATH, usecols=["query_location_id", "world_region"])
region_lookup = locations.set_index("query_location_id")["world_region"]
world_regions = sorted(locations["world_region"].dropna().unique())
if len(world_regions) != 5:
    raise ValueError(f"Expected five world regions, found {world_regions}")

## Aggregate regional flows

Read each completed allocation partition and reconcile its counts, shifted energy and weighted totals with the operator-year summary. Map sender and receiver locations to world regions and aggregate directed flows while retaining each setting, method, workload case and weather year.


In [19]:
region_parts = []
detail_columns = SCENARIO_METADATA_COLUMNS + [
    "allocation_method", "model_layer", "shift_mode", "operator_group", "weather_year",
    "sender_query_location_id", "receiver_query_location_id", "shifted_mwh",
    "weighted_renewable_gain", "weighted_abs_time_hours",
]

for (candidate_hours, shift_mode), partitions in allocation_inputs.items():
    for path, success in partitions:
        columns = detail_columns + (["candidate_hours"] if candidate_hours == 9 else [])
        frame = pd.read_parquet(path, columns=columns)
        if len(frame) != success["detail_row_count"] or not np.isclose(frame["shifted_mwh"].sum(), success["detail_shifted_mwh"], rtol=1e-12, atol=1e-6):
            raise ValueError(f"Allocation totals do not match the completion manifest: {path}")
        for column in ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode"]:
            if not frame[column].eq(success[column]).all():
                raise ValueError(f"Allocation identity {column} mismatch: {path}")
        if candidate_hours == 9 and not frame["candidate_hours"].eq(9).all():
            raise ValueError(f"Nine-hour detail contains another candidate-hour setting: {path}")
        frame["candidate_hours"] = candidate_hours
        frame = frame[frame["model_layer"].eq("current_plus_future")].copy()
        expected = base_operator.loc[
            base_operator["candidate_hours"].eq(candidate_hours)
            & base_operator["shift_mode"].eq(shift_mode)
            & base_operator["allocation_method"].eq(success["allocation_method"])
            & base_operator["utilisation_scenario_id"].eq(success["utilisation_scenario_id"])
            & base_operator["flexibility_scenario_id"].eq(success["flexibility_scenario_id"])
        ].set_index(["operator_group", "weather_year"])
        detail_totals = frame.groupby(["operator_group", "weather_year"]).agg(
            allocation_count=("shifted_mwh", "size"), shifted_mwh=("shifted_mwh", "sum"),
            total_weighted_renewable_gain=("weighted_renewable_gain", "sum"),
            total_weighted_abs_time_hours=("weighted_abs_time_hours", "sum"),
        )
        if len(detail_totals.index.difference(expected.index)):
            raise ValueError(f"Unexpected operator/weather-year allocation records: {path}")
        detail_totals = detail_totals.reindex(expected.index, fill_value=0)
        for column in detail_totals:
            if not np.allclose(detail_totals[column], expected[column], rtol=1e-12, atol=1e-5):
                raise ValueError(f"{candidate_hours}h summary/detail {column} mismatch: {path}")
        frame["sender_world_region"] = frame["sender_query_location_id"].map(region_lookup)
        frame["receiver_world_region"] = frame["receiver_query_location_id"].map(region_lookup)
        if frame[["sender_world_region", "receiver_world_region"]].isna().any().any():
            raise ValueError(f"Region lookup failed for {path}")
        group_cols = SCENARIO_METADATA_COLUMNS + [
            "candidate_hours", "allocation_method", "model_layer", "shift_mode", "weather_year",
            "sender_world_region", "receiver_world_region",
        ]
        region_parts.append(
            frame.groupby(group_cols, dropna=False)
            .agg(
                allocation_count=("shifted_mwh", "size"),
                baseline_shifted_mwh=("shifted_mwh", "sum"),
                baseline_total_weighted_renewable_gain=("weighted_renewable_gain", "sum"),
                baseline_total_weighted_abs_time_hours=("weighted_abs_time_hours", "sum"),
            )
            .reset_index()
        )
        del frame
    print(f"Aggregated {candidate_hours}h {shift_mode}: {len(partitions)} completed partitions")

if not region_parts:
    raise ValueError("No current-plus-future region allocations were found")

region_observed = pd.concat(region_parts, ignore_index=True)
region_group_cols = SCENARIO_METADATA_COLUMNS + [
    "candidate_hours", "allocation_method", "model_layer", "shift_mode", "weather_year",
    "sender_world_region", "receiver_world_region",
]
region_observed = region_observed.groupby(region_group_cols, dropna=False, as_index=False).agg(
    allocation_count=("allocation_count", "sum"),
    baseline_shifted_mwh=("baseline_shifted_mwh", "sum"),
    baseline_total_weighted_renewable_gain=("baseline_total_weighted_renewable_gain", "sum"),
    baseline_total_weighted_abs_time_hours=("baseline_total_weighted_abs_time_hours", "sum"),
)

Aggregated 6h daily_spatiotemporal: 18 completed partitions
Aggregated 6h same_hour_geographic: 18 completed partitions
Aggregated 9h daily_spatiotemporal: 18 completed partitions
Aggregated 9h same_hour_geographic: 18 completed partitions


## Complete the region-pair grid

Combine every validated case and weather year with all 25 directed region pairs. Fill unobserved flows and their allocation counts with zero so every pair is represented in the weather summaries.


In [20]:
base_keys = base_operator[SCENARIO_METADATA_COLUMNS + ["candidate_hours", "allocation_method", "model_layer", "shift_mode", "weather_year"]].drop_duplicates()
region_pairs = pd.DataFrame(product(world_regions, world_regions), columns=["sender_world_region", "receiver_world_region"])
region_base = base_keys.merge(region_pairs, how="cross").merge(region_observed, on=region_group_cols, how="left")
zero_columns = [
    "allocation_count", "baseline_shifted_mwh", "baseline_total_weighted_renewable_gain",
    "baseline_total_weighted_abs_time_hours",
]
region_base[zero_columns] = region_base[zero_columns].fillna(0)
region_base["allocation_count"] = region_base["allocation_count"].astype("int64")

## Scale regional flows

Apply each capacity factor to regional shifted energy and the weighted renewable and time totals. Recalculate the weighted metrics and leave them undefined where no energy is shifted.


In [21]:
region_weather = region_base.merge(projection_cross, how="cross")
scale = region_weather["capacity_scale_factor_vs_current_plus_future"]
region_weather["shifted_mwh"] = region_weather["baseline_shifted_mwh"] * scale
region_weather["shifted_gwh"] = region_weather["shifted_mwh"] / 1_000
region_weather["total_weighted_renewable_gain"] = region_weather["baseline_total_weighted_renewable_gain"] * scale
region_weather["total_weighted_abs_time_hours"] = region_weather["baseline_total_weighted_abs_time_hours"] * scale
region_weather["mwh_weighted_renewable_availability_gain"] = np.divide(
    region_weather["total_weighted_renewable_gain"], region_weather["shifted_mwh"],
    out=np.full(len(region_weather), np.nan), where=region_weather["shifted_mwh"].to_numpy() > 0,
)
region_weather["mwh_weighted_absolute_time_difference_hours"] = np.divide(
    region_weather["total_weighted_abs_time_hours"], region_weather["shifted_mwh"],
    out=np.full(len(region_weather), np.nan), where=region_weather["shifted_mwh"].to_numpy() > 0,
)
region_weather["is_primary_allocation_method"] = region_weather["allocation_method"].eq(PRIMARY_ALLOCATION_METHOD)

## Summarise regional weather sensitivity

Calculate median, minimum and maximum shifted energy across weather years for each directed region pair. Retain median base allocation counts and weighted renewable and time metrics.


In [22]:
region_summary_group_cols = projection_metadata + SCENARIO_METADATA_COLUMNS + [
    "candidate_hours", "allocation_method", "model_layer", "shift_mode", "sender_world_region", "receiver_world_region",
]
region = (
    region_weather.groupby(region_summary_group_cols, dropna=False)
    .agg(
        weather_year_count=("weather_year", "nunique"),
        shifted_gwh_median=("shifted_gwh", "median"),
        shifted_gwh_min_weather_year=("shifted_gwh", "min"),
        shifted_gwh_max_weather_year=("shifted_gwh", "max"),
        allocation_count_median=("allocation_count", "median"),
        mwh_weighted_renewable_availability_gain_median=("mwh_weighted_renewable_availability_gain", "median"),
        mwh_weighted_absolute_time_difference_hours_median=("mwh_weighted_absolute_time_difference_hours", "median"),
        weather_years_used=("weather_year", lambda values: "|".join(map(str, sorted(set(values))))),
    )
    .reset_index()
)
region["shifted_twh_median"] = region["shifted_gwh_median"] / 1_000
region["is_primary_allocation_method"] = region["allocation_method"].eq(PRIMARY_ALLOCATION_METHOD)

## Validate projected demand and scaling

Check expected result sizes, equal projected demand across hour settings and methods, and increasing shifted energy across the demand-growth scenarios. Confirm that uniform scaling preserves the shifted share within each allocation and workload case.


In [23]:
expected_rows = {
    "operator/weather-year": 3_888,
    "weather-year": 1_296,
    "summary": 432,
    "region-pair/weather-year": 32_400,
    "region-pair summary": 10_800,
}
actual_rows = {
    "operator/weather-year": len(operator_projected),
    "weather-year": len(weather),
    "summary": len(summary),
    "region-pair/weather-year": len(region_weather),
    "region-pair summary": len(region),
}
for name, expected in expected_rows.items():
    if actual_rows[name] != expected:
        raise ValueError(f"Expected {expected:,} {name} rows, found {actual_rows[name]:,}")

# Demand must remain identical across methods, and absolute projections must be monotonic by IEA scenario.
projected_demand_check = projected_demand.pivot_table(
    index=["projection_scenario_id", "utilisation_scenario_id", "weather_year"],
    columns=["candidate_hours", "allocation_method"], values="projected_total_demand_mwh", aggfunc="first",
)
if projected_demand_check.isna().any().any() or (projected_demand_check.max(axis=1) - projected_demand_check.min(axis=1)).max() > 1e-6:
    raise ValueError("Projected demand differs by candidate hours or allocation method")
scenario_pivot = summary.pivot_table(
    index=["candidate_hours", "allocation_method", "target_year", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode"],
    columns="demand_scenario_id", values="shifted_gwh_median", aggfunc="first",
)
if ((scenario_pivot["moderate"] + 1e-6 < scenario_pivot["conservative"]) | (scenario_pivot["accelerated"] + 1e-6 < scenario_pivot["moderate"])).any():
    raise ValueError("Projected shifted energy is not monotonic across demand-growth scenarios")

# Uniform scaling must preserve the method-specific shifted share of projected demand.
share_spread = summary.groupby([
    "candidate_hours", "allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode",
])["shifted_share_of_projected_demand_median"].agg(lambda values: values.max() - values.min()).max()
if share_spread > 1e-12:
    raise ValueError("Uniform projection changed a method-specific shifted demand share")

## Reconcile regional and total energy

Check that every projected case and weather year has all 25 directed region pairs and that their shifted energy sums to the corresponding total. Reconcile individual weather years because separate regional medians need not sum to the median total.


In [24]:
# Every projected method/case/year has a complete 5 × 5 region grid and reconciles before weather medians.
region_key_cols = [
    "candidate_hours", "allocation_method", "projection_scenario_id", "utilisation_scenario_id",
    "flexibility_scenario_id", "shift_mode", "weather_year",
]
if not region_weather.groupby(region_key_cols).size().eq(25).all():
    raise ValueError("The projected world-region grid is incomplete")
region_totals = region_weather.groupby(region_key_cols, as_index=False)["shifted_mwh"].sum()
weather_totals = weather[region_key_cols + ["shifted_mwh"]]
reconciliation = region_totals.merge(weather_totals, on=region_key_cols, suffixes=("_regions", "_total"), validate="one_to_one")
if (reconciliation["shifted_mwh_regions"] - reconciliation["shifted_mwh_total"]).abs().max() > 1e-5:
    raise ValueError("Projected region-pair flows do not reconcile to method totals")

## Check complete projected identities

Check that every expected projection, hour setting, method, workload case and weather year appears exactly once in the relevant table. Apply the same check to the operator and region-pair dimensions.


In [25]:
# Exact identities are checked separately from aggregate row counts.
projected_domains = {**case_domains, "projection_scenario_id": sorted(projection_cross["projection_scenario_id"])}
require_grid(operator_projected, {**projected_domains, "operator_group": OPERATORS, "weather_year": WEATHER_YEARS}, "Projected operator/weather years")
require_grid(weather, {**projected_domains, "weather_year": WEATHER_YEARS}, "Projected weather years")
require_grid(summary, projected_domains, "Projected summaries")

region_domains = {**projected_domains, "sender_world_region": world_regions, "receiver_world_region": world_regions}
require_grid(region_weather, {**region_domains, "weather_year": WEATHER_YEARS}, "Projected region/weather years")
require_grid(region, region_domains, "Projected region summaries")

## Check weighted totals and invariant ratios

Reconcile regional allocation counts and weighted renewable and time totals with the weather-year summaries. Check that uniform scaling preserves weighted metrics, same-hour-to-daily ratios and the ratio of nine-hour to six-hour shifted energy.


In [26]:
# Verify all region objective numerators and baseline flow counts, not only MWh.
for column in ["allocation_count", "total_weighted_renewable_gain", "total_weighted_abs_time_hours"]:
    region_values = region_weather.groupby(region_key_cols)[column].sum()
    weather_values = weather.set_index(region_key_cols)[column].reindex(region_values.index)
    if not np.allclose(region_values, weather_values, rtol=1e-12, atol=1e-5):
        raise ValueError(f"Region totals do not reconcile for {column}")

# Check the same-hour/daily ratio and weighted metrics remain invariant under scaling.
invariant_keys = ["candidate_hours", "allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode", "weather_year"]
for column in ["same_hour_share_of_daily_spatiotemporal", "mwh_weighted_renewable_availability_gain", "mwh_weighted_absolute_time_difference_hours"]:
    spreads = weather.groupby(invariant_keys)[column].agg(lambda values: values.max() - values.min())
    if spreads.dropna().max() > 1e-10:
        raise ValueError(f"Uniform scaling changed {column}")

# Matched 6h/9h percentage uplift is unchanged across projection endpoints.
pair_keys = ["allocation_method", "utilisation_scenario_id", "flexibility_scenario_id", "shift_mode", "weather_year", "projection_scenario_id"]
paired = weather.pivot(index=pair_keys, columns="candidate_hours", values="shifted_mwh")
paired["nine_to_six_ratio"] = paired[9] / paired[6].replace(0, np.nan)
uplift_spread = paired.reset_index().groupby(pair_keys[:-1])["nine_to_six_ratio"].agg(lambda values: values.max() - values.min())
if uplift_spread.dropna().max() > 1e-12:
    raise ValueError("Uniform scaling changed the paired six-/nine-hour uplift")

## Export validated projections

Save the IEA capacity factors and the validated operator, weather-year and regional projection tables. Export each result for six hours, nine hours and both settings together, retaining the workload reference and primary-method flags.


In [27]:
output_tables = [
    (operator_projected, OPERATOR_OUTPUT_PATH), (weather, WEATHER_OUTPUT_PATH),
    (summary, SUMMARY_OUTPUT_PATH), (region_weather, REGION_WEATHER_OUTPUT_PATH),
    (region, REGION_OUTPUT_PATH),
]
for directory in {PROJECTION_SCENARIOS_PATH.parent, NINE_HOUR_FINAL_DIR, COMBINED_FINAL_DIR, *(path.parent for _, path in output_tables)}:
    directory.mkdir(parents=True, exist_ok=True)

scenarios.to_csv(PROJECTION_SCENARIOS_PATH, index=False)
for frame, six_hour_path in output_tables:
    six_hour = frame.loc[frame["candidate_hours"].eq(6)].drop(columns=["candidate_hours"])
    nine_hour = frame.loc[frame["candidate_hours"].eq(9)]
    nine_hour_path = NINE_HOUR_FINAL_DIR / six_hour_path.name
    combined_path = COMBINED_FINAL_DIR / f"candidate_hours_{six_hour_path.name}"
    six_hour.to_csv(six_hour_path, index=False)
    nine_hour.to_csv(nine_hour_path, index=False)
    frame.to_csv(combined_path, index=False)
    print(f"{six_hour_path.name}: {len(six_hour):,} six-hour, {len(nine_hour):,} nine-hour, {len(frame):,} combined rows")

projected_flexibility_operator_weather_year_summary.csv: 1,944 six-hour, 1,944 nine-hour, 3,888 combined rows
projected_flexibility_weather_year_summary.csv: 648 six-hour, 648 nine-hour, 1,296 combined rows
projected_flexibility_summary.csv: 216 six-hour, 216 nine-hour, 432 combined rows
projected_flexibility_region_pair_weather_year_summary.csv: 16,200 six-hour, 16,200 nine-hour, 32,400 combined rows
projected_flexibility_region_pair_summary.csv: 5,400 six-hour, 5,400 nine-hour, 10,800 combined rows
